# ML Classification — Support Ops Copilot

This notebook trains two ticket classification models:

- Category classification
- Priority classification

The models use TF-IDF text features with Linear SVM.

In [14]:
import os
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

## 1. Load Processed Datasets

In [15]:
category_train = pd.read_csv("../../datasets/processed/classification/category_train.csv")
category_val = pd.read_csv("../../datasets/processed/classification/category_validation.csv")
category_test = pd.read_csv("../../datasets/processed/classification/category_test.csv")

priority_train = pd.read_csv("../../datasets/processed/classification/priority_train.csv")
priority_val = pd.read_csv("../../datasets/processed/classification/priority_validation.csv")
priority_test = pd.read_csv("../../datasets/processed/classification/priority_test.csv")

print("Category:")
print("Train:", category_train.shape)
print("Validation:", category_val.shape)
print("Test:", category_test.shape)

print("\nPriority:")
print("Train:", priority_train.shape)
print("Validation:", priority_val.shape)
print("Test:", priority_test.shape)

Category:
Train: (20010, 3)
Validation: (4288, 3)
Test: (4289, 3)

Priority:
Train: (20010, 3)
Validation: (4288, 3)
Test: (4289, 3)


## 2. Prepare Training Data

The training and validation sets are combined before final model training.

The test set remains separate and is used only for final evaluation.

In [16]:
category_train_full = pd.concat([category_train, category_val], ignore_index=True)
priority_train_full = pd.concat([priority_train, priority_val], ignore_index=True)

print("Category training rows:", len(category_train_full))
print("Priority training rows:", len(priority_train_full))

Category training rows: 24298
Priority training rows: 24298


## 3. Category Classification

The model predicts the ticket category (`type`) from the ticket text.

In [17]:
category_model = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000)),
                           ("svm", LinearSVC(max_iter=2000, random_state=42, class_weight="balanced"))
                        ])

category_model.fit(category_train_full["text"],category_train_full["type"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](4,)","['Change','Incident','Problem','Request']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [18]:
category_predictions = category_model.predict(category_test["text"])
category_accuracy = accuracy_score(category_test["type"], category_predictions)

category_f1 = f1_score(category_test["type"], category_predictions, average="macro")

print(f"Category Accuracy: {category_accuracy:.4f}")
print(f"Category Macro F1: {category_f1:.4f}")

print("\nClassification Report:")
print(classification_report(category_test["type"], category_predictions))

Category Accuracy: 0.8424
Category Macro F1: 0.8510

Classification Report:
              precision    recall  f1-score   support

      Change       0.98      0.97      0.98       437
    Incident       0.80      0.83      0.81      1721
     Problem       0.65      0.60      0.62       902
     Request       0.99      1.00      0.99      1229

    accuracy                           0.84      4289
   macro avg       0.85      0.85      0.85      4289
weighted avg       0.84      0.84      0.84      4289



## 4. Priority Classification

The model predicts ticket priority from the ticket text.

In [19]:
priority_model = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=10000)),
                           ("svm", LinearSVC(max_iter=2000, random_state=42, class_weight="balanced"))
                           ])

priority_model.fit(priority_train_full["text"], priority_train_full["priority"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('svm', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['high','low','medium']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",10000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [20]:
priority_predictions = priority_model.predict(priority_test["text"])

priority_accuracy = accuracy_score(priority_test["priority"], priority_predictions)

priority_f1 = f1_score(priority_test["priority"], priority_predictions, average="macro")

print(f"Priority Accuracy: {priority_accuracy:.4f}")
print(f"Priority Macro F1: {priority_f1:.4f}")

print("\nClassification Report:")
print(classification_report(priority_test["priority"], priority_predictions))

Priority Accuracy: 0.5614
Priority Macro F1: 0.5458

Classification Report:
              precision    recall  f1-score   support

        high       0.60      0.61      0.60      1677
         low       0.45      0.47      0.46       884
      medium       0.58      0.56      0.57      1728

    accuracy                           0.56      4289
   macro avg       0.55      0.55      0.55      4289
weighted avg       0.56      0.56      0.56      4289



## 5. Model Comparison

In [21]:
results = pd.DataFrame({"Task": ["Category", "Priority"],
                        "Accuracy": [category_accuracy, priority_accuracy],
                        "Macro F1": [category_f1, priority_f1]
                    })

results

,Task,Accuracy,Macro F1
0,Category,0.842388,0.850992
1,Priority,0.561436,0.545841


## 6. Save Trained Models

In [22]:
os.makedirs("../models/trained_models", exist_ok=True)

joblib.dump(category_model, "../models/trained_models/category_classifier.joblib")
joblib.dump(priority_model, "../models/trained_models/priority_classifier.joblib")

print("Models saved successfully.")

Models saved successfully.


## Final Results

### Category Classification
- TF-IDF + Linear SVM
- Accuracy: 84.24%
- Macro F1: 0.8510

### Priority Classification
- TF-IDF + Linear SVM
- Accuracy: 56.14%
- Macro F1: 0.5458

The category classifier performs substantially better than the priority classifier, indicating that priority labels are more difficult to predict reliably from ticket text alone.